In [0]:
# Read JSON array format (multiLine JSON)
df = spark.read.option("multiLine", "true").json("/Volumes/databricks_simulated_retail_customer_data/v01/retail-pipeline/orders/stream_json/")
df.display()

## The desired pipeline structure we are attempting to create is here
![image_1775431627249.png](./image_1775431627249.png "image_1775431627249.png")

In [0]:
spark.sql(f'''
          SELECT * 
          FROM JSON.`/Volumes/workspace/default/test_volume/test_directory/orders/`
          ''').display()

### Set this batch job to be run every day
(this is not incremental job as the table is not streaming table but static - meaning that each batch will update and process the entire table)

In [0]:
%sql
-- JSON -> Bronze
CREATE OR REPLACE TABLE `data-pipeline`.default.orders_bronze
AS 
SELECT *,
  current_timestamp() AS processing_tinme,
  _metadata.file_name AS source_file
FROM read_files(
  '/Volumes/workspace/default/test_volume/test_directory/orders/',
  format => 'json'
);
    
-- Bronze -> Silver
-- read the entire bronze table each time
CREATE OR REPLACE TABLE `data-pipeline`.default.orders_silver
AS
SELECT
  order_id,
  timestamp(order_timestamp) AS order_timestamp, 
  customer_id,
  notifications
FROM `data-pipeline`.default.orders_bronze;   

-- Silver -> Gold
-- Aggregate the silver each time the query is executed.
CREATE OR REPLACE VIEW `data-pipeline`.default.orders_by_date_vw     
AS 
SELECT 
  date(order_timestamp) AS order_date, 
  count(*) AS total_daily_orders
FROM `data-pipeline`.default.orders_silver                               
GROUP BY date(order_timestamp);

In [0]:
%sql
Select * from `data-pipeline`.default.orders_bronze limit 5

In [0]:
%sql
Select * from `data-pipeline`.default.orders_silver limit 5

In [0]:
%sql
Select * from `data-pipeline`.default.orders_by_date_vw limit 5

### Set the previous batch pipeline as a Lakeflow Declarative Pipelines

In [0]:
%sql
SELECT * 
FROM read_files(
  '/Volumes/workspace/default/test_volume/test_directory/orders/01.json',
  format => 'json'
)

In [0]:
%sql
SELECT * FROM `data-pipeline`.bronze_db.orders_bronze

In [0]:
spark.sql(f'list "/Volumes/workspace/default/test_volume/test_directory/orders/" ').display()

## Deploy a Pipeline to Production

![image_1775789583381.png](./image_1775789583381.png "image_1775789583381.png")

In [0]:
%sql
-- join the order and status table
with orders as (
  select *
  from read_files('/Volumes/workspace/default/test_volume/test_directory/orders/',
  format => 'json')
),
status as (
  select *
  from read_files('/Volumes/workspace/default/test_volume/test_directory/status/',
  format => 'json')
) -- join the views to get the order history with status
select 
  orders.order_id,
  timestamp(orders.order_timestamp) as order_timestamp,
  status.order_status,
  timestamp(status.status_timestamp) as order_status_timestamp
from orders
  inner join status
  on orders.order_id = status.order_id
order by order_id, order_status_timestamp